# 04 interactive option pricer

In [1]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from pricer import mc_pricing, bs_pricing
from gbm import simulate_paths

## parameter sliders

In [2]:
Sof0_slider  = widgets.FloatSlider(value=100, min=50,   max=200,  step=1,    description='S₀ (price)')
K_slider     = widgets.FloatSlider(value=105, min=50,   max=200,  step=1,    description='K (strike)')
r_slider     = widgets.FloatSlider(value=0.05, min=0.0, max=0.15, step=0.01, description='r (rate)')
sigma_slider = widgets.FloatSlider(value=0.20, min=0.05,max=0.60, step=0.01, description='σ (vol)')
T_slider     = widgets.FloatSlider(value=1.0,  min=0.1, max=3.0,  step=0.1,  description='T (years)')

for slider in [Sof0_slider, K_slider, r_slider, sigma_slider, T_slider]:
    slider.style.description_width = '100px'
    slider.layout.width = '500px'

## live pricer

In [3]:
output = widgets.Output()

def update(change):
    Sof0  = Sof0_slider.value
    K     = K_slider.value
    r     = r_slider.value
    sigma = sigma_slider.value
    T     = T_slider.value
    N     = int(252 * T)
    M     = 5000

    mc_price = mc_pricing(Sof0, K, r, sigma, T, N, M, option_type='call', seed=42)
    bs_price = bs_pricing(Sof0, K, r, sigma, T, option_type='call')

    paths       = simulate_paths(Sof0, r, sigma, T, N, M, seed=42)
    final       = paths[-1, :]
    payoffs     = np.maximum(final - K, 0)
    time        = np.linspace(0, T, N + 1)
    pct_worthless = (payoffs == 0).mean() * 100

    with output:
        output.clear_output(wait=True)

        fig, axes = plt.subplots(1, 3, figsize=(16, 4))

        # --- plot 1: fan chart ---
        axes[0].plot(time, paths[:, :200], color='steelblue', linewidth=0.3, alpha=0.15)
        axes[0].plot(time, paths[:, :200].mean(axis=1), color='black', linewidth=1.2, label='Mean')
        axes[0].axhline(Sof0, color='gray', linestyle='--', linewidth=0.8)
        axes[0].axhline(K, color='red', linestyle='--', linewidth=0.8, label=f'Strike K={K:.0f}')
        axes[0].set_title('Price paths')
        axes[0].set_xlabel('Time (years)')
        axes[0].set_ylabel('Stock price ($)')
        axes[0].legend(fontsize=8)

        # --- plot 2: payoff distribution ---
        axes[1].hist(payoffs[payoffs > 0], bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
        axes[1].set_title(f'Payoffs  |  {pct_worthless:.1f}% worthless')
        axes[1].set_xlabel('Payoff at expiry ($)')
        axes[1].set_ylabel('Frequency')

        # --- plot 3: convergence ---
        M_vals  = [50, 100, 500, 1000, 2000, 5000]
        mc_vals = [mc_pricing(Sof0, K, r, sigma, T, int(252*T), m, seed=42) for m in M_vals]
        axes[2].plot(M_vals, mc_vals, color='steelblue', linewidth=1.2, marker='o', markersize=4, label='MC estimate')
        axes[2].axhline(bs_price, color='black', linestyle='--', linewidth=1.0, label=f'BS = ${bs_price:.3f}')
        axes[2].set_xscale('log')
        axes[2].set_title('Convergence')
        axes[2].set_xlabel('M (simulations)')
        axes[2].set_ylabel('Option price ($)')
        axes[2].legend(fontsize=8)

        plt.suptitle(
            f'MC price: ${mc_price:.3f}   |   BS price: ${bs_price:.3f}   |   Diff: ${abs(mc_price - bs_price):.3f}',
            fontsize=11, y=1.02
        )
        plt.tight_layout()
        plt.show()

for slider in [Sof0_slider, K_slider, r_slider, sigma_slider, T_slider]:
    slider.observe(update, names='value')

display(Sof0_slider, K_slider, r_slider, sigma_slider, T_slider, output)
update(None)

FloatSlider(value=100.0, description='S₀ (price)', layout=Layout(width='500px'), max=200.0, min=50.0, step=1.0…

FloatSlider(value=105.0, description='K (strike)', layout=Layout(width='500px'), max=200.0, min=50.0, step=1.0…

FloatSlider(value=0.05, description='r (rate)', layout=Layout(width='500px'), max=0.15, step=0.01, style=Slide…

FloatSlider(value=0.2, description='σ (vol)', layout=Layout(width='500px'), max=0.6, min=0.05, step=0.01, styl…

FloatSlider(value=1.0, description='T (years)', layout=Layout(width='500px'), max=3.0, min=0.1, style=SliderSt…

Output()